In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# Load the dataset
data = pd.read_csv('dataset.csv')
# Display the first few rows of the dataset
data.head()

,ID,Sex,Age,Height,Weight,Hypertension,Diabetes,BMI,Level,Fitness Goal,Fitness Type,Exercises,Equipment,Diet,Recommendation
0,1,Male,18,1.68,47.5,No,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...",Dumbbells and barbells,"Vegetables: (Carrots, Sweet Potato, and Lettuc...",Follow a regular exercise schedule. Adhere to ...
1,2,Male,18,1.68,47.5,Yes,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...","Light athletic shoes, resistance bands, and li...","Vegetables: (Tomatoes, Garlic, leafy greens, b...",Follow a regular exercise schedule. Adhere to ...
2,3,Male,18,1.68,47.5,No,Yes,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, yoga, deadlifts, bench presses, and ov...","Dumbbells, barbells and Blood glucose monitor","Vegetables: (Garlic, Roma Tomatoes, Capers and...",Follow a regular exercise schedule. Adhere to ...
3,4,Male,18,1.68,47.5,Yes,Yes,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, yoga, deadlifts, bench presses, and ov...","Light athletic shoes, resistance bands, light ...","Vegetables: (Garlic, Roma Tomatoes, Capers, Gr...",Follow a regular exercise schedule. Adhere to ...
4,5,Male,18,1.68,47.5,No,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...",Dumbbells and barbells,"Vegetables: (Carrots, Sweet Potato, Lettuce); ...",Follow a regular exercise schedule. Adhere to ...


## Drop Unnecessary Columns

In [2]:
# drop Diet and Recommendation columns
data = data.drop(columns=['Diet', 'Recommendation'])

## Encode Categorical Data

In [3]:
from sklearn.preprocessing import LabelEncoder

# encode specified columns 'Sex', 'Hypertension', 'Diabetes', 'Fitness Goal', 'Fitness Type'
for column in ['Sex', 'Hypertension', 'Diabetes', 'Fitness Goal', 'Fitness Type']:
	data[column] = LabelEncoder().fit_transform(data[column])

# encode 'Level' column with custom mapping
level_mapping = {'Underweight': 0, 'Normal': 1, 'Overweight': 2, 'Obuse': 3}
data['Level'] = data['Level'].map(level_mapping)
data.head()

,ID,Sex,Age,Height,Weight,Hypertension,Diabetes,BMI,Level,Fitness Goal,Fitness Type,Exercises,Equipment
0,1,1,18,1.68,47.5,0,0,16.83,0,0,1,"Squats, deadlifts, bench presses, and overhead...",Dumbbells and barbells
1,2,1,18,1.68,47.5,1,0,16.83,0,0,1,"Squats, deadlifts, bench presses, and overhead...","Light athletic shoes, resistance bands, and li..."
2,3,1,18,1.68,47.5,0,1,16.83,0,0,1,"Squats, yoga, deadlifts, bench presses, and ov...","Dumbbells, barbells and Blood glucose monitor"
3,4,1,18,1.68,47.5,1,1,16.83,0,0,1,"Squats, yoga, deadlifts, bench presses, and ov...","Light athletic shoes, resistance bands, light ..."
4,5,1,18,1.68,47.5,0,0,16.83,0,0,1,"Squats, deadlifts, bench presses, and overhead...",Dumbbells and barbells


## Normalize Data

In [4]:
from sklearn.preprocessing import StandardScaler
# normalize Age, Height, Weight, BMI, Level columns
scaler = StandardScaler()
data[['Age', 'Height', 'Weight', 'BMI', 'Level']] = scaler.fit_transform(data[['Age', 'Height', 'Weight', 'BMI', 'Level']])
data.head()

,ID,Sex,Age,Height,Weight,Hypertension,Diabetes,BMI,Level,Fitness Goal,Fitness Type,Exercises,Equipment
0,1,1,-1.63391,-0.202298,-1.14858,0,0,-1.121606,-1.40336,0,1,"Squats, deadlifts, bench presses, and overhead...",Dumbbells and barbells
1,2,1,-1.63391,-0.202298,-1.14858,1,0,-1.121606,-1.40336,0,1,"Squats, deadlifts, bench presses, and overhead...","Light athletic shoes, resistance bands, and li..."
2,3,1,-1.63391,-0.202298,-1.14858,0,1,-1.121606,-1.40336,0,1,"Squats, yoga, deadlifts, bench presses, and ov...","Dumbbells, barbells and Blood glucose monitor"
3,4,1,-1.63391,-0.202298,-1.14858,1,1,-1.121606,-1.40336,0,1,"Squats, yoga, deadlifts, bench presses, and ov...","Light athletic shoes, resistance bands, light ..."
4,5,1,-1.63391,-0.202298,-1.14858,0,0,-1.121606,-1.40336,0,1,"Squats, deadlifts, bench presses, and overhead...",Dumbbells and barbells


In [5]:
# === MODEL: Multi-output classifier for Exercises & Equipment ===
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import joblib

# Fitur dan label
feature_cols = ['Sex','Age','Height','Weight','Hypertension','Diabetes','BMI','Level','Fitness Goal','Fitness Type']
X = data[feature_cols].values

# Encode target labels
le_ex = LabelEncoder().fit(data['Exercises'])
le_eq = LabelEncoder().fit(data['Equipment'])
y_ex = le_ex.transform(data['Exercises'])
y_eq = le_eq.transform(data['Equipment'])
y = np.column_stack([y_ex, y_eq])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
	X, y, test_size=0.2, random_state=42
)

# Model dasar
base = RandomForestClassifier(
	n_estimators=300,
	class_weight='balanced_subsample',
	n_jobs=-1,
	random_state=42
)
clf = MultiOutputClassifier(base)

# Train
clf.fit(X_train, y_train)

# Evaluasi
y_pred = clf.predict(X_test)
acc_ex = accuracy_score(y_test[:,0], y_pred[:,0])
acc_eq = accuracy_score(y_test[:,1], y_pred[:,1])
f1_ex = f1_score(y_test[:,0], y_pred[:,0], average='macro')
f1_eq = f1_score(y_test[:,1], y_pred[:,1], average='macro')
subset_acc = np.mean(np.all(y_pred == y_test, axis=1))
# sklearn.metrics.hamming_loss doesn't support multiclass-multioutput; compute manually
hamm = np.mean(y_test != y_pred)

print(f"Exercises  - Acc: {acc_ex:.3f} | F1-macro: {f1_ex:.3f}")
print(f"Equipment  - Acc: {acc_eq:.3f} | F1-macro: {f1_eq:.3f}")
print(f"Subset accuracy (both correct): {subset_acc:.3f}")
print(f"Hamming loss: {hamm:.3f}")

# Top-K rekomendasi per label
def top_k_recommendations(model, X_row, k=3):
	out = {}
	# Exercises
	est_ex = model.estimators_[0]
	if hasattr(est_ex, "predict_proba"):
		proba_ex = est_ex.predict_proba([X_row])[0]
		idx_ex = np.argsort(proba_ex)[::-1][:k]
		out["Exercises_topK"] = le_ex.inverse_transform(idx_ex).tolist()
	else:
		out["Exercises_topK"] = [le_ex.inverse_transform([est_ex.predict([X_row])[0]])[0]]
	# Equipment
	est_eq = model.estimators_[1]
	if hasattr(est_eq, "predict_proba"):
		proba_eq = est_eq.predict_proba([X_row])[0]
		idx_eq = np.argsort(proba_eq)[::-1][:k]
		out["Equipment_topK"] = le_eq.inverse_transform(idx_eq).tolist()
	else:
		out["Equipment_topK"] = [le_eq.inverse_transform([est_eq.predict([X_row])[0]])[0]]
	return out

# Contoh top-3 untuk sampel pertama test
example_rec = top_k_recommendations(clf, X_test[0], k=3)
print("Top-3:", example_rec)

# Simpan artefak untuk inferensi
artifacts = {
	"model": clf,
	"feature_cols": feature_cols,
	"scaler": scaler,  # dari sel normalisasi Anda
	"le_exercises": le_ex,
	"le_equipment": le_eq,
}
joblib.dump(artifacts, "recommender.joblib")


Exercises  - Acc: 0.998 | F1-macro: 0.998
Equipment  - Acc: 0.954 | F1-macro: 0.908
Subset accuracy (both correct): 0.954
Hamming loss: 0.024
Top-3: {'Exercises_topK': ['Brisk walking, cycling, swimming, running , or dancing.', 'brisk walking, cycling, swimming, or dancing.', 'Walking, Yoga, Swimming.'], 'Equipment_topK': ['Ellipticals, Indoor Rowers,Treadmills, and Rowing machine', 'Ellipticals, Indoor Rowers,Treadmills, Rowing machine', 'Light athletic shoes, resistance bands, light dumbbells and a Blood glucose monitor.']}


['recommender.joblib']